# LSM Regression Comparison

This notebook compares American option pricing under LSM using different continuation regressors.
We keep the setup small and simple for a quick comparison.


In [1]:
import pathlib, sys
repo_root = pathlib.Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pprint import pprint
from engines.monte_carlo import MonteCarloPricing


In [2]:
params = dict(S_0=100.0, X=100.0, sigma=0.2, T=1.0, r=0.05, num_paths=50_000, steps=50, seed=123)
pricer = MonteCarloPricing(**params)


In [3]:
results = []

# Polynomial bases
for basis in ["monomial", "laguerre", "hermite"]:
    price, stderr = pricer.american(call=True, basis_fn=basis, antithetic=True)
    results.append({"basis": basis, "price": price, "stderr": stderr})

# Neural network regressor
try:
    price, stderr = pricer.american(
        call=True,
        basis_fn="nn",
        antithetic=True,
        nn_kwargs={"hidden_sizes": (32, 32), "epochs": 50, "lr": 1e-3, "batch_size": 256},
    )
    results.append({"basis": "nn", "price": price, "stderr": stderr})
except ImportError as exc:
    results.append({"basis": "nn", "price": None, "stderr": None, "note": str(exc)})

pprint(results)


[{'basis': 'monomial',
  'price': 9.745166031812264,
  'stderr': 0.04931642746963015},
 {'basis': 'laguerre',
  'price': 10.073628305068903,
  'stderr': 0.05645604617296932},
 {'basis': 'hermite',
  'price': 10.159917497462686,
  'stderr': 0.058683520243318694},
 {'basis': 'nn', 'price': 10.443008997784254, 'stderr': 0.06525048314327425}]
